# 07 — Comprehensive Feature Selection from Lending Club

**Goal:** Select the FINAL pool of best features from ALL Lending Club features.

**What we do:**
1. Load ALL features from Lending Club (not just 27)
2. Correlation analysis — remove redundant
3. Permutation importance — rank by importance
4. Select top-K features
5. Output: final feature list for production

**Prerequisites:**
- `data/lending_club.csv` with ALL LC columns
- Update `src/data/ingestion.py` to load all columns

**Run:** `docker compose exec airflow-scheduler jupyter notebook notebooks/07_feature_selection_comprehensive.ipynb --allow-root`

In [ ]:
import sys
from pathlib import Path

docker_path = Path('/opt/airflow')
if docker_path.exists():
    sys.path.insert(0, str(docker_path))
else:
    ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)

print('✅ Modules loaded')

## 1. Load ALL Lending Club Features

In [ ]:
# Load full LC dataset
lc_path = Path('data/lending_club.csv')
if not lc_path.exists():
    raise FileNotFoundError(f'❌ {lc_path} not found! Download from Kaggle first.')

print(f'📊 Loading {lc_path}...')
df_lc = pd.read_csv(lc_path, low_memory=False)

print(f'✅ Loaded: {df_lc.shape[0]:,} rows × {df_lc.shape[1]} columns')
print(f'\n📝 All {df_lc.shape[1]} Lending Club features:')
for i, col in enumerate(df_lc.columns, 1):
    print(f'   {i:2d}. {col} ({df_lc[col].dtype})')

## 2. Feature Preprocessing

In [ ]:
# Select potentially useful features (exclude IDs, dates, text, URLs)
exclude_patterns = ['id', 'url', 'desc', 'title', 'emp_title', 'addr_state', 'zip_code']
exclude_cols = [col for col in df_lc.columns if any(pat in col.lower() for pat in exclude_patterns)]

# Also exclude post-origination features (leakage)
leakage_cols = ['out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 
                'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee',
                'recoveries', 'collection_recovery_fee', 'last_pymnt_amnt',
                'last_pymnt_d', 'next_pymnt_d', 'last_credit_pull_d']

exclude_cols.extend(leakage_cols)

# Keep only pre-origination features
candidate_cols = [col for col in df_lc.columns if col not in exclude_cols and col != 'loan_status']

print(f'🔍 Candidate features: {len(candidate_cols)}')
print(f'   Excluded: {len(exclude_cols)} (IDs, leakage, text)')
print(f'\n📝 Candidate features:')
for i, col in enumerate(candidate_cols, 1):
    print(f'   {i:2d}. {col}')

# Prepare target variable
default_statuses = {'Charged Off', 'Default', 'Late (31-120 days)', 
                    'Does not meet the credit policy. Status:Charged Off'}
df_lc['is_default'] = df_lc['loan_status'].isin(default_statuses).astype(int)

print(f'\n🎯 Target: is_default')
print(f'   Default rate: {df_lc["is_default"].mean():.2%}')

## 3. Encode Categorical Features

In [ ]:
# Separate numeric and categorical
numeric_cols = [col for col in candidate_cols if df_lc[col].dtype in ['float64', 'int64']]
categorical_cols = [col for col in candidate_cols if df_lc[col].dtype == 'object']

print(f'📊 Numeric features: {len(numeric_cols)}')
print(f'📊 Categorical features: {len(categorical_cols)}')

# Prepare dataframe
df_model = df_lc[numeric_cols + ['is_default']].copy()

# Encode categorical (target encoding for now)
for col in categorical_cols:
    if df_lc[col].nunique() < 50:  # Only low-cardinality
        # Target encoding
        target_map = df_lc.groupby(col)['is_default'].mean()
        df_model[col + '_target_enc'] = df_lc[col].map(target_map)
        print(f'   ✅ Encoded: {col} → {col}_target_enc ({df_lc[col].nunique()} categories)')
    else:
        print(f'   ⚠️  Skipped: {col} (too many categories: {df_lc[col].nunique()})')

# Fill NaN
df_model = df_model.fillna(df_model.median())

print(f'\n✅ Model ready: {df_model.shape[0]:,} rows × {df_model.shape[1]} features')

## 4. Correlation Analysis (Remove Redundant)

In [ ]:
# Correlation matrix
print('🔍 Computing correlation matrix...')
feature_cols = [col for col in df_model.columns if col != 'is_default']
corr_matrix = df_model[feature_cols].corr()

# Find highly correlated pairs (>0.85)
high_corr = []
for i in range(len(corr_matrix.columns)):
    for j in range(i):
        if abs(corr_matrix.iloc[i, j]) > 0.85:
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            high_corr.append({
                'feature1': col1,
                'feature2': col2,
                'correlation': corr_matrix.iloc[i, j]
            })

high_corr_df = pd.DataFrame(high_corr).sort_values('correlation', ascending=False)

print(f'\n📊 Highly Correlated Pairs (|r| > 0.85):')
if len(high_corr_df) > 0:
    print(high_corr_df.head(20).to_string(index=False))
    print(f'\n💡 Found {len(high_corr_df)} highly correlated pairs')
else:
    print('   None')

# Plot top correlations
if len(high_corr_df) > 0:
    plt.figure(figsize=(12, 8))
    plt.barh(range(min(20, len(high_corr_df))), high_corr_df['correlation'].abs().head(20)[::-1])
    plt.yticks(range(min(20, len(high_corr_df))), 
               [f"{r['feature1']} ↔ {r['feature2']}" for _, r in high_corr_df.head(20).iterrows()][::-1])
    plt.xlabel('|Correlation|')
    plt.title('Top 20 Highly Correlated Feature Pairs')
    plt.tight_layout()
    plt.show()

## 5. Permutation Importance (Rank All Features)

In [ ]:
# Train simple model for feature importance
print('🔧 Training model for feature importance...')

X = df_model.drop('is_default', axis=1)
y = df_model['is_default']

# Use Random Forest for fast feature importance
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X, y)

# Permutation importance (PR-AUC)
print('🔍 Computing permutation importance (PR-AUC)...')
result = permutation_importance(
    model, X, y,
    n_repeats=10,
    random_state=42,
    scoring='average_precision'
)

perm_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': result.importances_mean,
    'std': result.importances_std
}).sort_values('importance', ascending=False)

print(f'\n📊 Top 30 Features by Permutation Importance (PR-AUC):')
print(perm_importance.head(30).to_string(index=False))

# Plot top 30
plt.figure(figsize=(12, 12))
plt.barh(perm_importance['feature'][:30][::-1], 
         perm_importance['importance'][:30][::-1],
         xerr=perm_importance['std'][:30][::-1], 
         color='steelblue')
plt.xlabel('Permutation Importance (PR-AUC drop)')
plt.title('Top 30 Features — Permutation Importance')
plt.tight_layout()
plt.show()

## 6. Select Final Feature Pool

In [ ]:
# Remove redundant features (one from each highly correlated pair)
redundant = set()
for _, row in high_corr_df.iterrows():
    feat1, feat2 = row['feature1'], row['feature2']
    imp1 = perm_importance[perm_importance['feature'] == feat1]['importance'].values[0]
    imp2 = perm_importance[perm_importance['feature'] == feat2]['importance'].values[0]
    
    # Keep more important, mark other as redundant
    if imp1 < imp2:
        redundant.add(feat1)
    else:
        redundant.add(feat2)

# Select top features (non-redundant, importance > threshold)
importance_threshold = perm_importance['importance'].median() * 0.1
top_features = perm_importance[
    (perm_importance['importance'] >= importance_threshold) & 
    (~perm_importance['feature'].isin(redundant))
]['feature'].tolist()

print(f'🎯 FINAL FEATURE POOL: {len(top_features)} features')
print(f'   Removed: {len(redundant)} redundant')
print(f'   Threshold: {importance_threshold:.6f}')

print(f'\n📝 Final Features:')
for i, feat in enumerate(top_features, 1):
    imp = perm_importance[perm_importance['feature'] == feat]['importance'].values[0]
    print(f'   {i:2d}. {feat:<40} (importance: {imp:.6f})')

# Save to file
output_path = Path('artifacts/final_feature_pool.txt')
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    f.write('\n'.join(top_features))

print(f'\n✅ Saved to: {output_path}')

## 7. Summary

In [ ]:
print('='*80)
print('📋 COMPREHENSIVE FEATURE SELECTION SUMMARY')
print('='*80)

print(f'\n✅ Input:')
print(f'   Lending Club features: {df_lc.shape[1]}')
print(f'   Candidate features: {len(candidate_cols)}')
print(f'   Numeric: {len(numeric_cols)}, Categorical: {len(categorical_cols)}')

print(f'\n🔍 Analysis:')
print(f'   Highly correlated pairs: {len(high_corr_df)}')
print(f'   Redundant features removed: {len(redundant)}')

print(f'\n🎯 Output:')
print(f'   FINAL FEATURE POOL: {len(top_features)} features')
print(f'   Saved to: {output_path}')

print(f'\n💡 Next Steps:')
print('   1. Review final feature pool')
print('   2. Update src/config.py → FeatureConfig.ALL_FEATURES')
print('   3. Update src/data/ingestion.py → COLUMN_MAP')
print('   4. Update src/features/numerical.py → feature engineering')
print('   5. Re-run: data_ingestion → feature_engineering → model_training')
print('   6. Evaluate model performance')

print('\n' + '='*80)
print('✅ Comprehensive feature selection complete!')
print('='*80)